# VigilantDrive — Colab webcam demo

Runs the existing checkpoint in Colab and sends webcam frames from your browser. No training and no changes to Windows settings. Requires internet and browser camera permission. Select **Runtime → Change runtime type → T4 GPU** if available. Not a certified safety device.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import tempfile,zipfile
MYDRIVE=Path('/content/drive/MyDrive')
FOLDER=MYDRIVE/'VigilantDrive'/'VigilantDrive_from_scratch'
ZIP=MYDRIVE/'VigilantDrive'/'VigilantDrive_from_scratch.zip'
def valid(p): return (p/'config.yaml').is_file() and (p/'src/detector.py').is_file() and (p/'model/fcbam.py').is_file()
if valid(FOLDER): PROJECT=FOLDER
elif ZIP.is_file():
    unpack=Path(tempfile.mkdtemp(prefix='vd_demo_'))
    with zipfile.ZipFile(ZIP) as z: z.extractall(unpack)
    PROJECT=next((p.parent.parent for p in unpack.rglob('src/detector.py') if valid(p.parent.parent)),None)
    if PROJECT is None: raise FileNotFoundError('Project files were not found together inside the ZIP.')
else: raise FileNotFoundError(f'Put the project folder at {FOLDER} or ZIP at {ZIP}.')
CHECKPOINT=PROJECT/'runs/fcbam/weights/best.pt'
if not CHECKPOINT.is_file(): raise FileNotFoundError(f'best.pt missing at {CHECKPOINT}. Put your existing FCBAM best.pt there or edit CHECKPOINT to its Drive location.')
print('Project:',PROJECT,'\nCheckpoint:',CHECKPOINT)


In [ ]:
%pip -q install ultralytics==8.3.161 opencv-python-headless==4.11.0.86 PyYAML==6.0.2


In [ ]:
import sys,time,base64,cv2,numpy as np,torch,yaml
from IPython.display import display,Javascript,HTML,Image as IPImage
from google.colab.output import eval_js
sys.path.insert(0,str(PROJECT))
import ultralytics.nn.tasks as tasks
from model.fcbam import FCBAM
tasks.FCBAM=FCBAM
from src.detector import Detector,infer_eye_state,keep_driver_region
from src.drowsiness import DrowsinessMonitor
from src.metrics import Metrics
cfg=yaml.safe_load((PROJECT/'config.yaml').read_text(encoding='utf-8'))
device='0' if torch.cuda.is_available() else 'cpu'
print('PyTorch:',torch.__version__,'CUDA:',torch.cuda.is_available(),'device:',device)
detector=Detector(str(CHECKPOINT),device,cfg['confidence'],cfg['iou'],cfg.get('class_confidence'),cfg.get('imgsz',416),False,cfg.get('class_names',['Eyeclosed','Eyeopen','Yawn']))
monitor=DrowsinessMonitor(cfg)
metrics=Metrics('/content/drive/MyDrive/VigilantDrive/demo_results')
print('Session CSV:',metrics.path)


In [ ]:
display(Javascript('''
window.vdStart=async()=>{if(window.vdVideo&&window.vdVideo.srcObject)return true;const v=document.createElement('video');v.setAttribute('playsinline','');v.style.width='360px';v.style.maxWidth='100%';const s=await navigator.mediaDevices.getUserMedia({video:{facingMode:'user'},audio:false});v.srcObject=s;await v.play();document.body.appendChild(v);window.vdVideo=v;return true};
window.vdFrame=async()=>{const v=window.vdVideo;if(!v||!v.videoWidth)return null;const c=document.createElement('canvas');c.width=v.videoWidth;c.height=v.videoHeight;c.getContext('2d').drawImage(v,0,0);return c.toDataURL('image/jpeg',.7).split(',')[1]};
window.vdStop=()=>{if(window.vdVideo&&window.vdVideo.srcObject)window.vdVideo.srcObject.getTracks().forEach(t=>t.stop());window.vdVideo=null};
window.vdCtx=null;window.vdTimer=null;window.vdEnableAudio=async()=>{const A=window.AudioContext||window.webkitAudioContext;if(!A)return false;if(!window.vdCtx)window.vdCtx=new A();await window.vdCtx.resume();return true};
window.vdBeep=()=>{if(!window.vdCtx||window.vdCtx.state!=='running')return;const c=window.vdCtx,o=c.createOscillator(),g=c.createGain();o.frequency.value=950;g.gain.value=.12;o.connect(g);g.connect(c.destination);o.start();g.gain.setTargetAtTime(0,c.currentTime+.22,.02);o.stop(c.currentTime+.28)};
window.vdAlarmOn=()=>{if(!window.vdTimer){window.vdBeep();window.vdTimer=setInterval(window.vdBeep,650)}};window.vdAlarmOff=()=>{if(window.vdTimer)clearInterval(window.vdTimer);window.vdTimer=null};
'''))
display(HTML("<button onclick=\"window.vdEnableAudio().then(ok=>this.innerText=ok?'Alarm sound enabled':'Audio unavailable')\">Enable alarm sound</button>"))
print('Grant camera access when your browser asks.')
eval_js('window.vdStart()')


In [ ]:
# Start live inference; click the Colab stop button to end.
view=None;previous_alarm=False
try:
    while True:
        raw=eval_js('window.vdFrame()')
        if not raw: time.sleep(.1); continue
        frame=cv2.imdecode(np.frombuffer(base64.b64decode(raw),np.uint8),cv2.IMREAD_COLOR)
        if frame is None: continue
        t=time.monotonic();det=detector.detect(frame)
        det=keep_driver_region(det,frame.shape,cfg.get('driver_roi'))
        eye=infer_eye_state(det,float(cfg.get('single_eye_closed_confidence',.90)))
        yawn=any(n=='Yawn' for n,_,_ in det);monitor.update(t,eye,yawn);state=monitor.state()
        alarm=state=='DROWSY' and (monitor.eye_alarm_active() or monitor.yawn_alert_active())
        if alarm!=previous_alarm: eval_js('window.vdAlarmOn()' if alarm else 'window.vdAlarmOff()')
        previous_alarm=alarm
        fps=metrics.record(t,monitor.score(),monitor.perclos(),monitor.yawn_events(),state,detector.last_latency_ms,monitor.eye_status(),monitor.reason(),alarm,monitor.eye_coverage())
        color={'NORMAL':(50,200,50),'WARNING':(0,180,255),'DROWSY':(0,0,255)}[state]
        cv2.rectangle(frame,(0,0),(frame.shape[1],min(125,frame.shape[0])),(18,18,18),-1)
        cv2.putText(frame,f'{state} | {monitor.reason()}',(12,25),cv2.FONT_HERSHEY_SIMPLEX,.58,color,2)
        cv2.putText(frame,f'Eyes: {monitor.eye_status()} Coverage: {monitor.eye_coverage():.0%}',(12,51),cv2.FONT_HERSHEY_SIMPLEX,.44,(245,245,245),1)
        cv2.putText(frame,f'Score: {monitor.score():.2f} PERCLOS: {monitor.perclos():.2f} Yawns: {monitor.yawn_events()}',(12,77),cv2.FONT_HERSHEY_SIMPLEX,.44,(245,245,245),1)
        cv2.putText(frame,f'FPS: {fps:.1f} Inference: {detector.last_latency_ms:.0f} ms',(12,103),cv2.FONT_HERSHEY_SIMPLEX,.44,(245,245,245),1)
        if cfg.get('show_detection_boxes',True):
            palette={'Eyeclosed':(40,70,240),'Eyeopen':(60,190,70),'Yawn':(0,180,255)}
            for name,conf,(x1,y1,x2,y2) in det:
                c=palette.get(name,color);cv2.rectangle(frame,(x1,y1),(x2,y2),c,2);cv2.putText(frame,f'{name} {conf:.2f}',(x1,max(18,y1-4)),cv2.FONT_HERSHEY_SIMPLEX,.4,c,1)
        ok,jpg=cv2.imencode('.jpg',frame,[cv2.IMWRITE_JPEG_QUALITY,70])
        if ok:
            im=IPImage(data=jpg.tobytes())
            if view is None: view=display(im,display_id=True)
            else: view.update(im)
finally:
    eval_js('window.vdAlarmOff();window.vdStop();true')
    metrics.close()
    print('Stopped. Session CSV:',metrics.path)


## Notes

The browser streams frames to Colab, so latency depends on internet and runtime. If Colab does not assign a GPU, inference may be slow. The webcam/audio cells have not been verified on your exact account/device; test them before an interview. The local Windows block remains an OS policy issue and is not disabled by this notebook.